In [1]:
import sys, torch
print(sys.version)
print('torch', torch.__version__)
import transformers, accelerate
print('transformers', transformers.__version__)
print('accelerate', accelerate.__version__)
print('cuda', torch.cuda.is_available())
!nvidia-smi

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
torch 2.14.1+cu130
transformers 5.22.1
accelerate 1.15.0
cuda True
Thu Nov 19 04:00:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                     

In [2]:
%pip install -U "transformers>=5.0" accelerate
import transformers, accelerate, torch
print('transformers', transformers.__version__)
print('accelerate', accelerate.__version__)
print('torch', torch.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 28.4 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.15.0
    Uninstalling accelerate-1.15.0:
      Successfully uninstalled accelerate-1.15.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [accelerate]
transformers 5.22.1
accelerate 1.15.0
torch 2.14.1+cu130


In [1]:
import sys, torch
print(sys.version)
print('torch', torch.__version__)
import transformers, accelerate
print('transformers', transformers.__version__)
print('accelerate', accelerate.__version__)
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id='microsoft/Aion-1.0-Instruct-2.3B'
tokenizer=AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model=AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, dtype='auto', device_map='auto')
print('LOADED')
print('device', model.device)
print('dtype', model.dtype)
print('params_B', sum(p.numel() for p in model.parameters())/1e9)
print('gpu_mem_alloc_GB', torch.cuda.memory_allocated()/1e9)
print('gpu_mem_reserved_GB', torch.cuda.memory_reserved()/1e9)
print('gpu_max_mem_GB', torch.cuda.max_memory_allocated()/1e9)

def gen(user_text, max_new_tokens=256):
    messages=[{'role':'user','content':user_text}]
    text=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs=tokenizer(text, return_tensors='pt').to(model.device)
    print('input_tokens', inputs.input_ids.shape[1])
    out=model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new=out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True)

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
torch 2.14.1+cu130
transformers 5.23.0
accelerate 1.16.0


config.json:   0%|          | 0.00/1.90k [00:00<?, ?B/s]

configuration_aion_instruct.py:   0%|          | 0.00/10.2k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Aion-1.0-Instruct-2.3B:
- configuration_aion_instruct.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.5MB            

... (5 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

LOADED
device cuda:0
dtype torch.bfloat16
params_B 2.288831488
gpu_mem_alloc_GB 4.57767936
gpu_mem_reserved_GB 4.590665728
gpu_max_mem_GB 4.57767936


In [2]:
prompts={
'EN1': 'Give me a short introduction to large language models.',
'JP1': '次の文章を3行で要約してください。\n\n町立図書館は今年4月から、所蔵する郷土資料のデジタル化を始めた。対象は明治から昭和初期にかけての写真約1万2千点と、地元新聞の縮刷版320冊である。資料は劣化が進んでおり、このままでは10年以内に閲覧できなくなるものも多いという。デジタル化した資料は来年3月から館内の端末で無料公開される予定で、著作権の確認が取れたものについてはインターネット公開も検討している。事業費は総額4,800万円で、その半分を国の文化財保存補助金がまかなう。館長は『地域の記憶を次の世代に残すための最後の機会だ』と話している。',
'JP2': '次の文を、取引先に送る丁寧なビジネスメールの文面に書き換えてください。\n\nこの資料、明日までにざっと見といて。気になるとこあったらメモしといて。',
'JP3': '「猫」「雨」「傘」の3つの言葉をすべて使って、ちょうど3文で短い物語を書いてください。',
'JP4': '次の文章を読んで質問に答えてください。\n\n第3四半期の売上は前年同期比12%増の8億4,000万円だった。部門別では、クラウド事業が5億1,000万円で全体の約61%を占め、次いでサポート事業が2億2,000万円、ハードウェア事業が1億1,000万円だった。営業利益は9,600万円で、利益率は11.4%である。\n\n質問1: 売上が最も大きい部門とその金額は？ 質問2: 営業利益率は何%？',
'JP5': 'ある店では、りんご1個120円、みかん1個80円で売っています。りんごを3個とみかんを5個買うと合計はいくらですか。計算過程も書いてください。'
}
for k,v in prompts.items():
    print('##### '+k+' #####')
    print(gen(v,256))
    print('##### END '+k+' #####')

##### EN1 #####
input_tokens 49
Large language models (LLMs) are advanced artificial intelligence systems trained on vast amounts of text data to understand, generate, and manipulate human language. They can perform a wide range of tasks, such as answering questions, writing stories, translating languages, and assisting with coding, by predicting the next word in a sequence. Thanks to their size and training, LLMs can capture complex patterns in language and produce coherent, contextually relevant responses.
##### END EN1 #####
##### JP1 #####
input_tokens 254
町立図書館は4月から、明治～昭和初期の写真1万2千点と地元新聞縮刷版320冊をデジタル化。資料の劣化により10年以内に閲覧できなくなるものも多い。来年3月から館内端末で無料公開予定。国の文化財保存補助金で半額を賄う。
##### END JP1 #####
##### JP2 #####
input_tokens 95
以下は、ビジネスメールの文面として適切に書き換えたものです。

---

**件名：資料のご確認とご要望のご連絡**

〇〇様

この資料について、明日までにざっとご確認いただけますと幸いです。  
気になる点や、ご質問がございましたら、ぜひメモをいただけますと助かります。

何卒よろしくお願いいたします。

敬具  
[あなたの名前]  
[あなたの役職]  
[会社名]

---

### ポイント解説：
- **「ざっと見といて」→ 「ご確認いただけますと幸いです」**  
  取引先に対して、時間をかけずに見直してもらう丁寧な依頼

In [3]:
print(gen('Give me a short introduction to large language models.', 256))

input_tokens 49
Large language models (LLMs) are advanced artificial intelligence systems trained on vast amounts of text data to understand, generate, and manipulate human language. They can perform a wide range of tasks, such as answering questions, writing stories, translating languages, and assisting with coding, by predicting the next word in a sequence. Thanks to their size and training, LLMs can capture complex patterns in language and produce coherent, contextually relevant responses.


In [4]:
tools=[{'type':'function','function':{'name':'get_weather','description':'Get the current weather for a city.','parameters':{'type':'object','properties':{'city':{'type':'string','description':'City name'}},'required':['city']}}}]
msgs=[{'role':'user','content':'東京の今日の天気を教えてください。'}]
prompt_text=tokenizer.apply_chat_template(msgs, tools=tools, tokenize=False, add_generation_prompt=True)
print('=== CHAT_TEMPLATE_WITH_TOOLS ===')
print(prompt_text)
inputs=tokenizer(prompt_text, return_tensors='pt').to(model.device)
print('input_tokens', inputs.input_ids.shape[1])
out=model.generate(**inputs, max_new_tokens=256, do_sample=False)
raw=tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)
print('=== RAW_OUTPUT skip_special_tokens=False ===')
print(raw)
print('=== DECODED skip_special_tokens=True ===')
print(tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))
print('contains_tool_call_tag', '<|tool_call|>' in raw)

=== CHAT_TEMPLATE_WITH_TOOLS ===
<|system|>You are Aion Instruct, a large language model trained by Microsoft.
Knowledge cutoff: 2025-12
Current date: 2026-11-19

# Tools

You may call one or more functions to assist with the user query.

The available tools are provided as JSON between <|tool|> and <|/tool|> tokens:
<|tool|>[{"type": "function", "function": {"name": "get_weather", "description": "Get the current weather for a city.", "parameters": {"type": "object", "properties": {"city": {"type": "string", "description": "City name"}}, "required": ["city"]}}}]<|/tool|>

For each function call, respond with a single <|tool_call|>{"name": "<function-name>", "arguments": <json-args>}<|/tool_call|> block.
<|end|><|user|>東京の今日の天気を教えてください。<|end|><|assistant|>
input_tokens 180
=== RAW_OUTPUT skip_special_tokens=False ===
<|tool_call|>{"name": "get_weather", "arguments": {"city": "Tokyo"}}<|/tool_call|><|end|>
=== DECODED skip_special_tokens=True ===
<|tool_call|>{"name": "get_weather", "arg

In [5]:
filler='春の公園では桜が咲き、子どもたちが走り回っている。図書館の窓からは穏やかな光が差し込み、静かな時間が流れている。駅前の商店街では季節の野菜や果物が並び、買い物客でにぎわっている。\n'
needle='合言葉は『青いイルカ42番』です。\n'
question='合言葉は何ですか。文中の言葉で答えてください。'
print('filler_tokens', len(tokenizer(filler).input_ids))
print('needle_tokens', len(tokenizer(needle).input_ids))
def run_long(target):
    n=max(1,int(target/70))
    txt=''
    for _ in range(4):
        ctx=filler*n + needle + filler*2
        user=ctx+'\n質問: '+question
        txt=tokenizer.apply_chat_template([{'role':'user','content':user}], tokenize=False, add_generation_prompt=True)
        nt=len(tokenizer(txt).input_ids)
        print(f'target {target} try n={n} input_tokens={nt}')
        if abs(nt-target)<300:
            break
        n=int(n*target/nt)
    inputs=tokenizer(txt, return_tensors='pt').to(model.device)
    print('FINAL input_tokens', inputs.input_ids.shape[1])
    out=model.generate(**inputs, max_new_tokens=64, do_sample=False)
    ans=tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print('ANSWER:', ans)
    print('CONTAINS_NEEDLE', '青いイルカ42番' in ans)
    print('gpu_mem_alloc_GB', torch.cuda.memory_allocated()/1e9)
    return ans
for t in [4000,12000,27000]:
    print('##### LONG', t, '#####')
    try:
        run_long(t)
    except Exception as e:
        print('ERROR', type(e).__name__, str(e))
    print('##### END LONG', t, '#####')

filler_tokens 80
needle_tokens 14
##### LONG 4000 #####
target 4000 try n=57 input_tokens=4793
target 4000 try n=47 input_tokens=3993
FINAL input_tokens 3993
ANSWER: 青いイルカ42番
CONTAINS_NEEDLE True
gpu_mem_alloc_GB 4.586300928
##### END LONG 4000 #####
##### LONG 12000 #####
target 12000 try n=171 input_tokens=13913
target 12000 try n=147 input_tokens=11993
FINAL input_tokens 11993
ERROR OutOfMemoryError CUDA out of memory. Tried to allocate 12.86 GiB. GPU 0 has a total capacity of 14.56 GiB of which 7.44 GiB is free. Including non-PyTorch memory, this process has 7.12 GiB memory in use. Of the allocated memory 5.71 GiB is allocated by PyTorch, and 1.28 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
##### END LONG 12

In [6]:
import torch, transformers, accelerate
torch.cuda.empty_cache()
print('transformers', transformers.__version__)
print('accelerate', accelerate.__version__)
print('torch', torch.__version__)
print('gpu_mem_alloc_GB', torch.cuda.memory_allocated()/1e9)
print(gen('こんにちは。', 32))

transformers 5.23.0
accelerate 1.16.0
torch 2.14.1+cu130
gpu_mem_alloc_GB 4.58620416
input_tokens 41
こんにちは！今日はどのようなお手伝いが必要ですか？何か質問や相談があれば、いつでもお気軽にどうぞ。


In [7]:
import gc, torch
print('before del gpu_mem_alloc_GB', torch.cuda.memory_allocated()/1e9)
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print('after free gpu_mem_alloc_GB', torch.cuda.memory_allocated()/1e9)
print('after free gpu_mem_reserved_GB', torch.cuda.memory_reserved()/1e9)

before del gpu_mem_alloc_GB 4.58620416
after free gpu_mem_alloc_GB 0.0085248
after free gpu_mem_reserved_GB 4.580179968


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id='Qwen/Qwen2.5-3B-Instruct'
tokenizer=AutoTokenizer.from_pretrained(model_id)
model=AutoModelForCausalLM.from_pretrained(model_id, dtype='auto', device_map='auto')
print('QWEN LOADED')
print('device', model.device)
print('dtype', model.dtype)
print('params_B', sum(p.numel() for p in model.parameters())/1e9)
print('gpu_mem_alloc_GB', torch.cuda.memory_allocated()/1e9)
print('gpu_mem_reserved_GB', torch.cuda.memory_reserved()/1e9)
print('attn_implementation', model.config._attn_implementation)
print('model_type', model.config.model_type)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

QWEN LOADED
device cuda:0
dtype torch.bfloat16
params_B 3.085938688
gpu_mem_alloc_GB 6.21186048
gpu_mem_reserved_GB 6.268387328
attn_implementation sdpa
model_type qwen2


In [9]:
prompts={
'Q-EN1': 'Give me a short introduction to large language models.',
'Q-JP1': '次の文章を3行で要約してください。\n\n町立図書館は今年4月から、所蔵する郷土資料のデジタル化を始めた。対象は明治から昭和初期にかけての写真約1万2千点と、地元新聞の縮刷版320冊である。資料は劣化が進んでおり、このままでは10年以内に閲覧できなくなるものも多いという。デジタル化した資料は来年3月から館内の端末で無料公開される予定で、著作権の確認が取れたものについてはインターネット公開も検討している。事業費は総額4,800万円で、その半分を国の文化財保存補助金がまかなう。館長は『地域の記憶を次の世代に残すための最後の機会だ』と話している。',
'Q-JP2': '次の文を、取引先に送る丁寧なビジネスメールの文面に書き換えてください。\n\nこの資料、明日までにざっと見といて。気になるとこあったらメモしといて。',
'Q-JP3': '「猫」「雨」「傘」の3つの言葉をすべて使って、ちょうど3文で短い物語を書いてください。',
'Q-JP4': '次の文章を読んで質問に答えてください。\n\n第3四半期の売上は前年同期比12%増の8億4,000万円だった。部門別では、クラウド事業が5億1,000万円で全体の約61%を占め、次いでサポート事業が2億2,000万円、ハードウェア事業が1億1,000万円だった。営業利益は9,600万円で、利益率は11.4%である。\n\n質問1: 売上が最も大きい部門とその金額は？ 質問2: 営業利益率は何%？',
'Q-JP5': 'ある店では、りんご1個120円、みかん1個80円で売っています。りんごを3個とみかんを5個買うと合計はいくらですか。計算過程も書いてください。'
}
for k,v in prompts.items():
    print('##### '+k+' #####')
    print(gen(v,256))
    print('##### END '+k+' #####')

##### Q-EN1 #####
input_tokens 39
Large Language Models (LLMs) are advanced artificial intelligence systems designed to understand and generate human-like text. These models are trained on vast amounts of textual data, allowing them to learn the nuances of natural language, including grammar, syntax, semantics, and context.

Key characteristics of LLMs include:

1. **Scale**: They are typically trained on extremely large datasets, often in the billions or trillions of words, which allows them to capture a wide range of linguistic patterns and contexts.

2. **Contextual Understanding**: LLMs can process and generate text that is coherent and relevant to the given context, even when presented with new or unusual inputs.

3. **Generative Capabilities**: Beyond understanding, these models can also generate text, such as writing stories, emails, articles, or even code snippets, based on prompts or specific instructions.

4. **Adaptability**: They can be fine-tuned for various tasks, such as

In [10]:
tools=[{'type':'function','function':{'name':'get_weather','description':'Get the current weather for a city.','parameters':{'type':'object','properties':{'city':{'type':'string','description':'City name'}},'required':['city']}}}]
msgs=[{'role':'user','content':'東京の今日の天気を教えてください。'}]
prompt_text=tokenizer.apply_chat_template(msgs, tools=tools, tokenize=False, add_generation_prompt=True)
print('=== CHAT_TEMPLATE_WITH_TOOLS ===')
print(prompt_text)
inputs=tokenizer(prompt_text, return_tensors='pt').to(model.device)
print('input_tokens', inputs.input_ids.shape[1])
out=model.generate(**inputs, max_new_tokens=256, do_sample=False)
raw=tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)
print('=== RAW_OUTPUT skip_special_tokens=False ===')
print(raw)
print('=== DECODED skip_special_tokens=True ===')
print(tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))
print('contains_tool_call', '<tool_call>' in raw or '<|tool_call|>' in raw)

=== CHAT_TEMPLATE_WITH_TOOLS ===
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_weather", "description": "Get the current weather for a city.", "parameters": {"type": "object", "properties": {"city": {"type": "string", "description": "City name"}}, "required": ["city"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
東京の今日の天気を教えてください。<|im_end|>
<|im_start|>assistant

input_tokens 182
=== RAW_OUTPUT skip_special_tokens=False ===
<tool_call>
{"name": "get_weather", "arguments": {"city": "東京"}}
</tool_call><|im_end|>
=== DECODED skip_special_tokens=True =

In [11]:
import gc, torch
print('before del gpu_mem_alloc_GB', torch.cuda.memory_allocated()/1e9)
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print('after free gpu_mem_alloc_GB', torch.cuda.memory_allocated()/1e9)
print('after free gpu_mem_reserved_GB', torch.cuda.memory_reserved()/1e9)
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id='microsoft/Aion-1.0-Instruct-2.3B'
tokenizer=AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model=AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, dtype='auto', device_map='auto', attn_implementation='sdpa')
print('AION SDPA LOADED')
print('device', model.device)
print('dtype', model.dtype)
print('params_B', sum(p.numel() for p in model.parameters())/1e9)
print('gpu_mem_alloc_GB', torch.cuda.memory_allocated()/1e9)
print('gpu_mem_reserved_GB', torch.cuda.memory_reserved()/1e9)
print('attn_implementation', model.config._attn_implementation)
print('model_type', model.config.model_type)

before del gpu_mem_alloc_GB 6.21186048
after free gpu_mem_alloc_GB 0.0085248
after free gpu_mem_reserved_GB 4.580179968


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

AION SDPA LOADED
device cuda:0
dtype torch.bfloat16
params_B 2.288831488
gpu_mem_alloc_GB 4.58620416
gpu_mem_reserved_GB 4.59276288
attn_implementation sdpa
model_type aion_instruct


In [12]:
print('filler_tokens', len(tokenizer(filler).input_ids))
print('needle_tokens', len(tokenizer(needle).input_ids))
print('attn_implementation', model.config._attn_implementation)
print('##### LONG 12000 SDPA RETRY #####')
try:
    run_long(12000)
except Exception as e:
    print('ERROR', type(e).__name__, str(e))
print('##### END LONG 12000 SDPA RETRY #####')

filler_tokens 80
needle_tokens 14
attn_implementation sdpa
##### LONG 12000 SDPA RETRY #####
target 12000 try n=171 input_tokens=13913
target 12000 try n=147 input_tokens=11993
FINAL input_tokens 11993
ERROR OutOfMemoryError CUDA out of memory. Tried to allocate 12.86 GiB. GPU 0 has a total capacity of 14.56 GiB of which 8.60 GiB is free. Including non-PyTorch memory, this process has 5.96 GiB memory in use. Of the allocated memory 5.72 GiB is allocated by PyTorch, and 121.76 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
##### END LONG 12000 SDPA RETRY #####
